# Pattern — ReAct

**ReAct** (Reason + Act) interleaves **reasoning**, **tool actions**, and **observations** in a loop. The agent decides at runtime what to do next based on what it has learned so far — unlike a fixed prompt chain or a single router decision.

```
  User question
       │
       ▼
  ┌─────────┐     ┌─────────┐     ┌─────────────┐
  │ Thought │ ──► │ Action  │ ──► │ Observation │
  └────┬────┘     └─────────┘     └──────┬──────┘
       │                                  │
       └──────── scratchpad ◄─────────────┘
       │
       ▼ (when enough evidence)
  Final Answer
```

Each cycle appends to a **scratchpad** — the growing transcript of thoughts, actions, and tool results that informs the next step.

This notebook implements ReAct for **ShopCo Support Hub** and **ReleaseFlow** in plain Python — tool registry, scratchpad loop, step budget, and full trace logging.

In [ ]:
"""
ReAct Pattern Lab — ShopCo + ReleaseFlow.
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

USE_LIVE_LLM = False
MAX_REACT_STEPS = 8


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "customer": "alice@shop.com", "status": "shipped", "total_usd": 1299.0},
    "ORD-1002": {"order_id": "ORD-1002", "customer": "bob@shop.com", "status": "processing", "total_usd": 449.0},
}

POLICIES: dict[str, str] = {
    "returns": "30-day returns on unused items. [pol-returns-01]",
    "refunds": "Cite policy before promising refunds. Escalate over $1000. [pol-refunds-03]",
}

BILLING: dict[str, dict[str, Any]] = {
    "ORD-1001": {"refund_eligible": True},
    "ORD-1002": {"refund_eligible": True},
}

RELEASE_CHECKS: dict[str, dict[str, str]] = {
    "unit_tests": {"status": "pass", "detail": "412 tests passed"},
    "security_scan": {"status": "warn", "detail": "1 medium CVE in dev dependency"},
    "lint": {"status": "pass", "detail": "no issues"},
}

REACT_TRACE: list[dict[str, Any]] = []

print("ReAct lab ready")

---

## 1. ReAct vs Chain vs Router

| Pattern | Steps decided by | Tool use |
|---------|------------------|----------|
| **Prompt chain** | Developer (fixed) | Predetermined |
| **Router** | Classifier once | One handler |
| **ReAct** | Agent each iteration | Dynamic, multi-step |

ReAct shines when the **path to the answer is unknown** — the agent must discover which tools to call and in what order.

---

## 2. Tool Registry — ShopCo Actions

In [ ]:
@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: list[str]
    handler: Callable[[dict[str, Any]], str]


def lookup_order(args: dict[str, Any]) -> str:
    oid = args.get("order_id", "")
    order = ORDERS.get(oid)
    if not order:
        return f"Observation: order {oid} not found."
    return f"Observation: {oid} status={order['status']}, total=${order['total_usd']:.0f}."


def get_policy(args: dict[str, Any]) -> str:
    topic = args.get("topic", "returns")
    text = POLICIES.get(topic, POLICIES["returns"])
    return f"Observation: policy({topic}): {text}"


def check_billing(args: dict[str, Any]) -> str:
    oid = args.get("order_id", "")
    bill = BILLING.get(oid)
    if not bill:
        return f"Observation: no billing record for {oid}."
    return f"Observation: refund_eligible={bill['refund_eligible']}."


def create_escalation(args: dict[str, Any]) -> str:
    ticket = f"ESC-{uuid.uuid4().hex[:6].upper()}"
    return f"Observation: escalation ticket {ticket} created."


SHOPCO_TOOLS: dict[str, ToolSpec] = {
    "lookup_order": ToolSpec("lookup_order", "Fetch order status and total", ["order_id"], lookup_order),
    "get_policy": ToolSpec("get_policy", "Retrieve policy snippet", ["topic"], get_policy),
    "check_billing": ToolSpec("check_billing", "Check refund eligibility", ["order_id"], check_billing),
    "create_escalation": ToolSpec("create_escalation", "Open human escalation ticket", [], create_escalation),
}

print(f"ShopCo tools: {list(SHOPCO_TOOLS)}")

---

## 3. ReAct Step Model and Scratchpad

In [ ]:
class StepKind(str, Enum):
    THOUGHT = "thought"
    ACTION = "action"
    OBSERVATION = "observation"
    FINAL = "final"


@dataclass
class ReActStep:
    kind: StepKind
    content: str
    tool_name: str | None = None
    tool_args: dict[str, Any] = field(default_factory=dict)


@dataclass
class ReActRun:
    run_id: str
    question: str
    steps: list[ReActStep] = field(default_factory=list)
    final_answer: str = ""
    stopped_reason: str = ""

    def scratchpad(self) -> str:
        lines = []
        for s in self.steps:
            if s.kind == StepKind.THOUGHT:
                lines.append(f"Thought: {s.content}")
            elif s.kind == StepKind.ACTION:
                lines.append(f"Action: {s.tool_name}({json.dumps(s.tool_args)})")
            elif s.kind == StepKind.OBSERVATION:
                lines.append(s.content)
            elif s.kind == StepKind.FINAL:
                lines.append(f"Final Answer: {s.content}")
        return "\n".join(lines)


print("ReAct step model ready")

---

## 4. Offline Reasoning Policy

Production ReAct uses an LLM to emit Thought / Action / Final Answer lines. Here a **rule-based policy** simulates that behavior deterministically for offline demos.

In [ ]:
def extract_order_id(text: str) -> str | None:
    m = re.search(r"ORD-\d+", text.upper())
    return m.group() if m else None


class ShopCoReActPolicy:
    """Decides next thought + action from scratchpad state."""

    def next_move(self, question: str, steps: list[ReActStep]) -> tuple[str, str | None, dict[str, Any]]:
        """Returns (thought, tool_name | None, tool_args). tool_name=None means final answer."""
        oid = extract_order_id(question)
        is_refund = "refund" in question.lower()
        actions_taken = {s.tool_name for s in steps if s.kind == StepKind.ACTION}
        observations = " ".join(s.content for s in steps if s.kind == StepKind.OBSERVATION)

        if oid and "lookup_order" not in actions_taken:
            return (f"Need order facts for {oid}.", "lookup_order", {"order_id": oid})
        if is_refund and "get_policy" not in actions_taken:
            return ("Refund request — fetch refunds policy.", "get_policy", {"topic": "refunds"})
        if is_refund and oid and "check_billing" not in actions_taken:
            return ("Verify refund eligibility in billing.", "check_billing", {"order_id": oid})
        if is_refund and "total=$1299" in observations and "create_escalation" not in actions_taken:
            return ("High-value order — open escalation per policy.", "create_escalation", {})
        return ("Enough evidence to reply.", None, {})

    def compose_final(self, question: str, steps: list[ReActStep]) -> str:
        parts = []
        for s in steps:
            if s.kind == StepKind.OBSERVATION:
                parts.append(s.content.replace("Observation: ", ""))
        return " ".join(parts) if parts else "I could not find enough information."


policy = ShopCoReActPolicy()
print("ShopCo ReAct policy ready")

---

## 5. ReAct Engine — Thought → Action → Observation Loop

In [ ]:
class ReActEngine:
    def __init__(self, tools: dict[str, ToolSpec], max_steps: int = MAX_REACT_STEPS):
        self.tools = tools
        self.max_steps = max_steps

    def execute_tool(self, name: str, args: dict[str, Any]) -> str:
        spec = self.tools.get(name)
        if not spec:
            return f"Observation: unknown tool '{name}'."
        return spec.handler(args)

    def run(self, question: str, reasoning: ShopCoReActPolicy) -> ReActRun:
        run = ReActRun(str(uuid.uuid4())[:8], question)
        step_count = 0

        while step_count < self.max_steps:
            thought, tool_name, tool_args = reasoning.next_move(question, run.steps)
            run.steps.append(ReActStep(StepKind.THOUGHT, thought))

            if tool_name is None:
                answer = reasoning.compose_final(question, run.steps)
                run.steps.append(ReActStep(StepKind.FINAL, answer))
                run.final_answer = answer
                run.stopped_reason = "final_answer"
                break

            run.steps.append(ReActStep(StepKind.ACTION, "", tool_name=tool_name, tool_args=tool_args))
            obs = self.execute_tool(tool_name, tool_args)
            run.steps.append(ReActStep(StepKind.OBSERVATION, obs))
            step_count += 1
        else:
            run.final_answer = reasoning.compose_final(question, run.steps)
            run.stopped_reason = "max_steps"

        REACT_TRACE.append({
            "run_id": run.run_id, "steps": len(run.steps),
            "reason": run.stopped_reason, "ts": utcnow(),
        })
        return run


engine = ReActEngine(SHOPCO_TOOLS)
print("ReAct engine ready")

---

## 6. ShopCo Refund Ticket — Full ReAct Trace

In [ ]:
question = "I want a refund for ORD-1001"
run = engine.run(question, policy)

print(run.scratchpad())
print(f"\nStopped: {run.stopped_reason}")

---

## 7. Simple Lookup — Fewer Steps

In [ ]:
class SimpleLookupPolicy:
    def next_move(self, question: str, steps: list[ReActStep]) -> tuple[str, str | None, dict[str, Any]]:
        oid = extract_order_id(question)
        actions = {s.tool_name for s in steps if s.kind == StepKind.ACTION}
        if oid and "lookup_order" not in actions:
            return (f"Look up {oid}.", "lookup_order", {"order_id": oid})
        return ("Done.", None, {})

    def compose_final(self, question: str, steps: list[ReActStep]) -> str:
        for s in steps:
            if s.kind == StepKind.OBSERVATION:
                return s.content.replace("Observation: ", "")
        return "No data."


simple = engine.run("Where is ORD-1002?", SimpleLookupPolicy())
print(f"Steps: {len(simple.steps)} -> {simple.final_answer}")

---

## 8. Parsing LLM ReAct Output (Action Lines)

When using a live model, actions arrive as text. A parser extracts tool name and JSON args.

In [ ]:
ACTION_RE = re.compile(r"Action:\s*(\w+)\((.*)\)", re.DOTALL)
FINAL_RE = re.compile(r"Final Answer:\s*(.+)", re.DOTALL)


def parse_react_line(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("Thought:"):
        return {"kind": "thought", "content": text[8:].strip()}
    m = ACTION_RE.search(text)
    if m:
        name, args_str = m.group(1), m.group(2).strip()
        args = json.loads(args_str) if args_str else {}
        return {"kind": "action", "tool": name, "args": args}
    m = FINAL_RE.search(text)
    if m:
        return {"kind": "final", "content": m.group(1).strip()}
    return {"kind": "unknown", "content": text}


samples = [
    "Thought: I should look up the order.",
    'Action: lookup_order({"order_id": "ORD-1001"})',
    "Final Answer: Your order is shipped.",
]
for line in samples:
    print(parse_react_line(line))

---

## 9. ReleaseFlow ReAct — Ship Decision

In [ ]:
def run_tests(_: dict[str, Any]) -> str:
    c = RELEASE_CHECKS["unit_tests"]
    return f"Observation: tests {c['status']} — {c['detail']}."


def run_security(_: dict[str, Any]) -> str:
    c = RELEASE_CHECKS["security_scan"]
    return f"Observation: security {c['status']} — {c['detail']}."


def run_lint(_: dict[str, Any]) -> str:
    c = RELEASE_CHECKS["lint"]
    return f"Observation: lint {c['status']} — {c['detail']}."


RF_TOOLS = {
    "run_tests": ToolSpec("run_tests", "Execute unit test suite", [], run_tests),
    "run_security": ToolSpec("run_security", "Run security scan", [], run_security),
    "run_lint": ToolSpec("run_lint", "Run linter", [], run_lint),
}


class ReleaseReActPolicy:
    SEQUENCE = ["run_tests", "run_lint", "run_security"]

    def next_move(self, question: str, steps: list[ReActStep]) -> tuple[str, str | None, dict[str, Any]]:
        done = [s.tool_name for s in steps if s.kind == StepKind.ACTION]
        for tool in self.SEQUENCE:
            if tool not in done:
                return (f"Run {tool} before ship decision.", tool, {})
        return ("All checks complete.", None, {})

    def compose_final(self, question: str, steps: list[ReActStep]) -> str:
        obs = " ".join(s.content for s in steps if s.kind == StepKind.OBSERVATION)
        if "security warn" in obs.lower():
            return "NO-GO: security scan has warnings — resolve CVE before ship."
        return "GO: all checks passed."


rf_engine = ReActEngine(RF_TOOLS)
rf_run = rf_engine.run("Can we ship v2.4.1 to production?", ReleaseReActPolicy())
print(rf_run.scratchpad())
print(f"\nDecision: {rf_run.final_answer}")

---

## 10. Step Budget and Loop Guardrails

In [ ]:
class LoopingPolicy:
    """Buggy policy that always requests lookup_order — triggers max_steps guard."""

    def next_move(self, question: str, steps: list[ReActStep]) -> tuple[str, str | None, dict[str, Any]]:
        oid = extract_order_id(question) or "ORD-9999"
        return ("Keep looking...", "lookup_order", {"order_id": oid})

    def compose_final(self, question: str, steps: list[ReActStep]) -> str:
        return "Stopped — step budget exhausted."


guarded = ReActEngine(SHOPCO_TOOLS, max_steps=3)
loop_run = guarded.run("ORD-1001?", LoopingPolicy())
print(f"Stopped: {loop_run.stopped_reason}, action count={sum(1 for s in loop_run.steps if s.kind == StepKind.ACTION)}")

---

## 11. Duplicate Action Detection

In [ ]:
def detect_duplicate_actions(steps: list[ReActStep]) -> list[str]:
    seen: set[tuple[str, str]] = set()
    dupes: list[str] = []
    for s in steps:
        if s.kind != StepKind.ACTION or not s.tool_name:
            continue
        key = (s.tool_name, json.dumps(s.tool_args, sort_keys=True))
        if key in seen:
            dupes.append(f"{s.tool_name}{s.tool_args}")
        seen.add(key)
    return dupes


dupes = detect_duplicate_actions(loop_run.steps)
print(f"Duplicate actions in loop run: {dupes}")

---

## 12. Trace Inspection

In [ ]:
print(f"ReAct runs logged: {len(REACT_TRACE)}")
for entry in REACT_TRACE:
    print(f"  {entry['run_id']}: {entry['steps']} steps, {entry['reason']}")

---

## 13. When to Use ReAct

| Good fit | Poor fit |
|----------|----------|
| Unknown tool sequence | Fixed 2-step pipeline |
| Multi-hop lookup (order → policy → billing) | Single database query |
| Exploratory troubleshooting | Strict latency SLA with known path |
| Agent must adapt to observations | Deterministic ETL transform |

---

## 14. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| No step limit | Infinite tool loops | `max_steps` + duplicate detection |
| Vague tool descriptions | Wrong tool chosen | Clear names, params, examples |
| Hidden scratchpad | Cannot debug failures | Log thought/action/observation |
| Too many tools | Analysis paralysis | Curate tool set per agent |
| ReAct for fixed pipeline | Wasted reasoning tokens | Use prompt chain instead |
| Ignoring observation format | Parser breaks | Standardize Observation: prefix |

---

## 15. Check Your Understanding

1. What three elements cycle in ReAct?
2. How is ReAct different from a **prompt chain**?
3. What is the **scratchpad** for?
4. Why does ShopCo call `create_escalation` for ORD-1001 refunds?
5. What stops a run when the agent loops forever?
6. Why parse Action lines into structured tool calls?
7. When would ReleaseFlow return NO-GO?

---

## 16. Optional — Live LLM ReAct

Set `USE_LIVE_LLM = True` to drive the same engine with model-generated Thought / Action lines.

In [ ]:
if USE_LIVE_LLM:
    import os
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    tool_desc = "\n".join(f"- {t.name}: {t.description}" for t in SHOPCO_TOOLS.values())
    prompt = f"Tools:\n{tool_desc}\n\nUse Thought/Action/Observation until Final Answer."
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": "Refund for ORD-1001"},
        ],
    )
    print(resp.choices[0].message.content)
else:
    print("Offline mode — rule-based ReAct for ShopCo and ReleaseFlow.")

---

## 17. Summary

ReAct combines **reasoning** with **tool use** in a dynamic loop:

- **Thought** — plan next step from scratchpad
- **Action** — invoke a tool with structured args
- **Observation** — append tool result to context
- **Final Answer** — synthesize when evidence is sufficient

ShopCo refund tickets traverse order → policy → billing → escalation. ReleaseFlow runs tests, lint, and security before GO/NO-GO. Always cap steps, log the scratchpad, and prefer simpler patterns when the tool sequence is known upfront.